# Chapter 12 &mdash; A Transformer on a Language That Needs a Stack

**Concept 12 of the Chapter 12 decomposition:** *A Transformer on a Language That Needs a Stack*

Karpathy's baby GPT in the raw, trained on strings a Jove PDA accepts &mdash; and then judged by that same PDA.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12-PDA/Concept-Karpathy-GPT-On-A-Jove-PDA/Concept-Karpathy-GPT-On-A-Jove-PDA.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


In Chapter 6 this same 120 lines of PyTorch was trained on the strings a **DFA**
accepts, and the picture it turned into had the shape of the DFA. A transformer with
a context window of $k$ symbols *is* a finite-state machine over the $2^k$ windows, so
that was a fair fight.

This chapter's languages are not regular. **Balanced brackets** needs a stack, and a
stack has no bound. The model still only sees the last $k$ symbols.

So the question of this notebook is not "does the loss go down" &mdash; it will. It is:

> when the loss has gone down, **has the model learned the language?**

We are in the one chapter that can answer that properly, because Jove has the PDA.
Every string the model invents gets handed to `run_pda`, and the PDA says yes or no.
That is the notebook: train, sample, and let the machine mark the homework.

The answer is worth predicting before you run it.

## 2. Definitions

### The model &mdash; Karpathy's, unchanged

In [ ]:
#@title minimal GPT implementation in PyTorch  (Andrej Karpathy)
#
# Read it, change it, break it.  This is the whole model: an embedding, a
# few attention blocks, a linear head.  Nothing here knows about automata.
""" super minimal decoder-only gpt """

import math
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                    .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k ,v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # manual implementation of attention
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.c_proj(y)
        return y

class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.nonlin = nn.GELU()

    def forward(self, x):
        x = self.c_fc(x)
        x = self.nonlin(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

@dataclass
class GPTConfig:
    # these are default GPT-2 hyperparameters
    block_size: int = 1024
    vocab_size: int = 50304
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    bias: bool = False

class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %d" % (sum(p.nelement() for p in self.parameters()),))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        device = idx.device
        b, t = idx.size()
        assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
        pos = torch.arange(0, t, dtype=torch.long, device=device).unsqueeze(0) # shape (1, t)

        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (1, t, n_embd)
        x = tok_emb + pos_emb
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x[:, -1, :]) # note: only returning logits at the last time step (-1), output is 2D (b, vocab_size)
        return logits

### Cutting a sequence into examples, and the training loop

In [ ]:
def make_XY(seq, context_length):
    X, Y = [], []
    for i in range(len(seq) - context_length):
        X.append(seq[i:i + context_length])
        Y.append(seq[i + context_length])
    return (torch.tensor(X, dtype=torch.long),
            torch.tensor(Y, dtype=torch.long))

def train_gpt(gpt, X, Y, iters=200, lr=1e-3, every=20):
    optimizer = torch.optim.AdamW(gpt.parameters(), lr=lr, weight_decay=1e-1)
    losses = []
    for i in range(iters):
        logits = gpt(X)
        loss = F.cross_entropy(logits, Y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        losses.append(loss.item())
        if i % every == 0 or i == iters - 1:
            print(i, loss.item())
    return losses

### The corpus, from a PDA

In [ ]:
# --- the training data: strings the PDA accepts, run together ------------
from functools import reduce

def pda_accepts(P, s, STKMAX=20):
    surv, paths, visited = run_pda(s, P, acceptance='ACCEPT_F',
                                   STKMAX=STKMAX, chatty=False)
    return len(paths) > 0        # accepted iff SOME path ended in F

def corpus_from(P, upto=16000, STKMAX=8):
    strings = [nthnumeric(i, ['0', '1']) for i in range(upto)]
    good = [s for s in strings if pda_accepts(P, s, STKMAX)]
    return good, list(map(int, reduce(lambda a, b: a + b, good)))

### Sampling from a trained model

In [ ]:
# --- sample from the model, exactly as Karpathy does --------------------
def sample(gpt, start, steps=24):
    xi = list(start)
    full = xi.copy()
    for _ in range(steps):
        x = torch.tensor(xi, dtype=torch.long)[None, ...]
        probs = nn.functional.softmax(gpt(x), dim=-1)
        t = torch.multinomial(probs[0], num_samples=1).item()
        xi = xi[1:] + [t]
        full.append(t)
    return ''.join(map(str, full))

### The language

In [ ]:
# --- the language: balanced brackets, with 0 for '(' and 1 for ')' -------
# The stack is doing the one thing no DFA can do: counting with no bound.
DYCK = md2mc('''PDA
IF : 0 , # ; 0# -> A
A  : 0 , 0 ; 00 -> A
A  : 1 , 0 ; '' -> A
A  : '' , # ; # -> IF
''')

<!-- nav-strip -->

---

&larr;&nbsp;[Ch12&nbsp;11.&nbsp;A Tale of Three Parsers, and Why Textual Syntax Still Rules](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12-PDA/Concept-Tale-Of-Three-Parsers/Concept-Tale-Of-Three-Parsers.ipynb) &nbsp;&middot;&nbsp; [**Chapter 12** index](https://github.com/ganeshutah/Jove/blob/master/Chapter12-PDA/README.md) &nbsp;&middot;&nbsp; [Ch12&nbsp;13.&nbsp;More Context Is Not a Stack](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter12-PDA/Concept-More-Context-Is-Not-A-Stack/Concept-More-Context-Is-Not-A-Stack.ipynb)&nbsp;&rarr;

---

## 3. Tests

The PDA first. Check it against a hand-written bracket counter before trusting it to mark anything.

In [ ]:
def balanced(s):
    depth = 0
    for c in s:
        depth += 1 if c == '0' else -1
        if depth < 0: return False
    return depth == 0

from itertools import product
cases = [''.join(p) for n in range(9) for p in product('01', repeat=n)]
bad = [s for s in cases if pda_accepts(DYCK, s) != balanced(s)]
print('disagreements over %d strings :' % len(cases), bad)
assert not bad
print('the PDA and the counter agree -- the PDA can be trusted as judge')

**The corpus.** Every balanced string Jove can enumerate, run together.

In [ ]:
good, seq = corpus_from(DYCK)
print('accepted strings :', len(good))
print('first few        :', [s or 'eps' for s in good[:8]])
print('longest           :', max(map(len, good)))
print('training symbols  :', len(seq))

Three symbols in, the next symbol out &mdash; exactly as in Chapter 6.

In [ ]:
vocab_size, context_length = 2, 3
X, Y = make_XY(seq, context_length)
for i in range(6):
    print('example %2d: %s --> %s' % (i + 1, X[i].tolist(), Y[i].item()))
print(X.shape, Y.shape)

**Train**, and watch the loss fall. It falls.

In [ ]:
config = GPTConfig(block_size=context_length, vocab_size=vocab_size,
                   n_layer=4, n_head=4, n_embd=16, bias=False)
torch.manual_seed(1337)
gpt = GPT(config)
losses = train_gpt(gpt, X, Y, iters=200, every=25)

Below $\ln 2 = 0.693$, which is where a coin would sit. Something was learned. **Now let the PDA say what.**

In [ ]:
# --- generate a string, then let the PDA be the judge --------------------
# The seed is the same SHAPE for every context length -- a shallow prefix
# 0101... -- so that changing k does not quietly change the question.
def seed_for(k):
    return [int(ch) for ch in ('01' * k)[:k]]

def try_samples(gpt, k, P, length=16, n=30, seed=0):
    torch.manual_seed(seed)
    got = [sample(gpt, seed_for(k), steps=length - k) for _ in range(n)]
    ok = [s for s in got if pda_accepts(P, s)]
    return got, ok


got, ok = try_samples(gpt, context_length, DYCK, length=16, n=30)
print('generated 30 strings of length 16; the PDA accepts %d of them' % len(ok))
print()
for s in got[:8]:
    print('   %s   %s' % (s, 'ACCEPTED' if pda_accepts(DYCK, s) else 'rejected'))

So the loss fell and the language was **not** learned. Where exactly does it go wrong? A balanced string may never close a bracket that was never opened; ask the model how much probability it puts on doing precisely that.

In [ ]:
# --- how much probability does the model give a FORBIDDEN symbol? --------
# Where the stack is empty the language cannot close, so '1' is forbidden.
# Where a capped stack is full it cannot open, so '0' is forbidden.  Ask
# the model how much mass it puts there.  No threshold and no score: one
# number, between 0 and 1, saying how much of the rule it has picked up.
def forbidden_mass(gpt, k, good, cap=None):
    total, n = 0.0, 0
    for s in good:
        depth = 0
        for i, c in enumerate(s):
            bad = 1 if depth == 0 else (0 if depth == cap else None)
            if bad is not None and i >= k:
                x = torch.tensor([int(ch) for ch in s[i - k:i]],
                                 dtype=torch.long)[None, ...]
                p = nn.functional.softmax(gpt(x), dim=-1)[0].tolist()
                total += p[bad]
                n += 1
            depth += 1 if c == '0' else -1
    return total / max(n, 1), n


torch.manual_seed(1337)
before = GPT(config)                 # an untrained copy, for comparison
m0, n = forbidden_mass(before, context_length, good)
m1, _ = forbidden_mass(gpt, context_length, good)
print('at the %d places in the corpus where the language FORBIDS closing:' % n)
print('   untrained model gives that forbidden symbol  %.3f of the mass' % m0)
print('   trained   model gives that forbidden symbol  %.3f of the mass' % m1)

Read that as the honest summary of the notebook.

In [ ]:
print("It started at a half, which is what a coin does, and it came down.")
print("It did not come down to zero, and it will not.")
print()
print("To know whether closing a bracket is legal you must know whether the")
print("stack is empty, and that is a fact about the WHOLE prefix -- every")
print("symbol since the start.  The model is shown three.  No amount of")
print("training puts information into a window that never carried it.")
print()
print("This is Concept 5 of this chapter, arrived at from the other side:")
print("the pumping lemma says no finite memory suffices; here is a finite")
print("memory failing to suffice, measured in probability mass.")

## 4. Animation

The machine that marked the homework.

In [ ]:
from jove.AnimatePDA import *
AnimatePDA(DYCK, FuseEdges=True)

## 5. Exercises


1. Predict, before running: if you raise `context_length` to 5, does the forbidden
   mass go to zero, halve, or barely move? The next notebook measures it.
2. The corpus is every balanced string up to length 12, run end to end. The join
   points are strings the language never contains. How could you avoid that, and
   would it help?
3. `try_samples` seeds with `0101...`. Seed with `000` instead and re-run. Does the
   acceptance rate change, and is that a fact about the model or about the seed?
4. Train for 2000 iterations instead of 200. Report the loss and the forbidden mass.
5. Chapter 6's DFA notebook could draw the model as a machine with $2^3$ states.
   That drawing is still legal here &mdash; the model has not changed. What has
   changed is what it is being asked to imitate. Say in one sentence why the
   drawing is no longer a picture of the language.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter12-PDA/Concept-Karpathy-GPT-On-A-Jove-PDA')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')